# #4 Tracing performance

## Purpose

Evaluate tracing performance in the full dataset

## Setup

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import networkx as nx
import numpy as np
import pandas as pd
import pycea as py
import seaborn as sns
import treedata as td
from devmap.config import (
    embryo_palette,
    embryos,
    get_paths,
    lineage_palette,
    phase_palette,
    set_theme,
    stage_palette,
    type_palette,
)
from devmap.topology import marked_branches_by_depth
from devmap.utils import load_data, save_plot
from numba import njit

set_theme()
base_path, plots_path, results_path = get_paths("validation")

## Load data

In [ ]:
tdata = load_data('topology', characters = True)
characters = tdata.obsm["characters"].copy()
cell_counts = pd.read_csv(base_path / "data" / "external" / "Bondarenko_2023_cell_counts.csv")

## PE expression vs edit fraction

In [119]:
tdata.obs["pe_expr"] = np.log((tdata.obs["pe_counts"] / tdata.obs["total_counts"]) * 2e4 + 1)
df = tdata.obs.query("type == 'donor'").groupby(["cell_type","stage"]).agg(
    {"pe_counts": "mean", "edit_frac": "mean","pe_expr": "mean"}).reset_index()
fig, ax = plt.subplots(figsize=(2, 2), dpi=600, layout="constrained")
sns.scatterplot(data = df,x = "pe_expr",y = "edit_frac", hue = "stage", palette=stage_palette, ax = ax, s = 10)
plt.ylabel("Cell type mean edit fraction")
plt.xlabel("Cell type mean PE2maxGFP expression")
plt.xlim(0, 4)
plt.ylim(0, 0.7)
save_plot(plots_path / "pe_expr_vs_edit_frac.svg")

## Relative abundance

In [186]:
type_counts = tdata.obs.groupby(["cell_subtype","type","lineage"]).size()
type_counts = type_counts.unstack("type").fillna(0).reset_index()
type_counts["donor"] = type_counts["donor"] + 1
type_counts["host"] = type_counts["host"] + 1
fig, ax = plt.subplots(figsize=(2, 2), dpi=600)
plt.plot([0.6, 1e6], [0.6, 1e6], color="gray", linestyle="--", linewidth=1)
sns.scatterplot(data=type_counts, x="host", y="donor", hue="lineage", legend = False, s = 10, palette = lineage_palette)
plt.ylim(.6, 2e5)
plt.xlim(.6, 2e5)
plt.xscale("log")
plt.yscale("log")
for axis in [ax.xaxis, ax.yaxis]:
    axis.set_major_locator(mticker.LogLocator(base=10.0, numticks=100))
    axis.set_minor_locator(mticker.LogLocator(base=10.0, subs='auto', numticks=100))
plt.ylabel("Donor cell count")
plt.xlabel("Host cell count")
save_plot(plots_path / "donor_host_subtype_counts.svg", fig, transparent=True)

## Chimerism rate

In [11]:
tdata.obs["embryo"] = pd.Categorical(tdata.obs["embryo"], categories=reversed(embryos), ordered=True)
counts = (
    tdata.obs
    .groupby(['embryo', 'type'], observed=False)
    .size()
    .unstack(fill_value=0)
)
percent = counts.div(counts.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(1.2,2.5), dpi=600)
percent.plot(kind='barh', stacked=True, color = type_palette, ax = ax,edgecolor='black', linewidth=.5)
# remove legend
ax.legend().remove()
ax.set_xlabel("Percent")
ax.set_xticks([0, 50, 100])
plt.tight_layout()
save_plot(plots_path / "embryo_type_composition.svg", fig, transparent=True)

## Detection rate

In [225]:
fig, ax = plt.subplots(figsize=(1.5,2.5), dpi=600)
obs = tdata.obs.copy()
obs["detection_pct"] = obs["detection_rate"] * 100
sns.boxplot(obs.query("clone.notnull()").sample(100000), y = "embryo", x = "detection_pct", linecolor="black", linewidth=.6,
    order = embryos, showfliers = False, hue = "stage", palette=stage_palette, ax = ax, legend = False,saturation = 1,
    medianprops={"linewidth": 1})
# add star at mean for each embryo
means = obs.query("clone.notnull()").groupby("embryo")["detection_pct"].median()
plt.xlabel("Detection rate (%)") 
plt.ylabel("")
plt.xlim(60,105)
save_plot(plots_path / "embryo_detection_rate.svg", fig, transparent=True)

## Character vs tree distance

In [156]:
@njit
def norm_hamming_distance(arr1, arr2):
    valid_mask = (arr1 != -1) & (arr2 != -1)
    hamming_distance = 0
    for x, y in zip(arr1[valid_mask], arr2[valid_mask]):
        if x == y:
            pass
        elif x == 0 or y == 0:
            hamming_distance += 1
        else:
            hamming_distance += 2
    num_valid_comparisons = np.sum(valid_mask)
    if num_valid_comparisons == 0:
        return 0
    normalized_distance = hamming_distance / num_valid_comparisons
    return normalized_distance

py.tl.tree_distance(tdata,sample_n=20000, depth_key="time", update = False)
py.tl.distance(tdata, metric=norm_hamming_distance, key="characters", connect_key="tree_connectivities", update=False)

In [180]:
df = py.tl.compare_distance(tdata, dist_keys = ["tree","characters"]).query("obs1 != obs2")
df["tree_distances"] = df["tree_distances"] / 2
df["stage"] = df["obs1"].str.split("-").str[0]
fig, ax = plt.subplots(figsize=(2, 2), dpi=600)
sns.scatterplot(data=df.sample(frac=1), x="tree_distances", y="characters_distances",
                alpha=0.5, hue = "stage", palette = stage_palette, s = 5, ax = ax)
plt.xticks([0,2,4,6,8,10])
plt.xlabel("Tree Distance (Days Since LCA)")
plt.ylabel("Normalized Hamming Distance")
save_plot(plots_path / "tree_vs_character_distance.svg", fig, transparent=True, rasterize=True)

## Number of extant cells over time

In [3]:
n_extant = py.tl.n_extant(tdata,bins = np.arange(0,10.5,.5),copy = True, depth_key="time")
n_extant["embryo"] = n_extant["tree"].str.split("-C").str[0]
n_extant = n_extant.groupby(['time','embryo']).agg({'n_extant':'sum'}).reset_index()
n_extant["timepoint"] = n_extant["embryo"].str.split("-").str[0].str.replace("E", "").astype(float)
n_extant["stage"] = n_extant["embryo"].str.split("-R").str[0]

All embryos

In [28]:
cell_counts = cell_counts.query("Category == 'In vivo'").copy()
cell_counts["time"] = cell_counts["Stage"].str.replace("E", "").astype(float)

In [27]:
fig, ax = plt.subplots(figsize=(2,2),dpi=600, layout = "constrained")
sns.lineplot(data=n_extant.query("time <= timepoint"), x="time", y="n_extant", hue="stage", legend = False, 
             linewidth = 1, palette = stage_palette)
sns.boxplot(data = cell_counts, x = "time", y = "EPI", ax=ax,native_scale=True, 
            showfliers = False, linewidth = .5, medianprops={"linewidth": 1},
            color = "black",
            boxprops={
            "facecolor": "none",
            "edgecolor": "black",
            "linewidth": 0.5,
    },)
plt.xticks([0,2,4,6,8,10])
plt.xlabel("Embryonic time (days)")
plt.ylabel("Number of extant cells")
plt.yscale('log')
save_plot(plots_path / "growth_kinetics.svg", fig)

E9.5

In [ ]:
fig, ax = plt.subplots(figsize=(1.9,1.9),dpi=600, layout = "constrained")
sns.lineplot(data=n_extant.query("stage == 'E9.5' & time <= timepoint"), x="time", y="n_extant", hue="embryo", palette=embryo_palette, legend = False)
plt.xticks([0,2,4,6,8,10])
plt.xlabel("Embryonic time (days)")
plt.ylabel("Number of extant cells")
plt.yscale('log')
save_plot(plots_path / "e9.5_growth_kinetics.svg", fig)

## Fraction of branches marked by an edit

In [ ]:
marked_branches = []
for clone, tree in tdata.obst.items():
    df = marked_branches_by_depth(tree, bins=[0,2,4,6,7,8,9,10,11])
    marked_branches.append(df.assign(clone = clone))
marked_branches = pd.concat(marked_branches, ignore_index=True)

All embryos

In [ ]:
fig, ax = plt.subplots(figsize=(2,1.8),dpi=600, layout = "constrained")
marked_branches["embryo"] = marked_branches["clone"].str.split("-C").str[0]
marked_branches["stage"] = marked_branches["clone"].str.split("-").str[0]
sns.lineplot(data=marked_branches, x="bin_center", y="pct_marked", hue = "stage",palette=stage_palette,legend = False, linewidth = 1)
plt.ylabel("Resolved branches (%)")
plt.xticks([0,2,4,6,8,10])
plt.yticks([60,70,80,90,100])
plt.ylim(55,105)
plt.xlabel("Embryonic time (days)")
save_plot(plots_path / "marked_branches.svg", fig)

E9.5

In [ ]:
fig, ax = plt.subplots(figsize=(2,1.8),dpi=600, layout = "constrained")
marked_branches["embryo"] = marked_branches["clone"].str.split("-C").str[0]
sns.lineplot(data=marked_branches.query("stage == 'E9.5'"), x="bin_center", y="pct_marked", hue = "embryo",palette=embryo_palette,legend = False)
plt.ylabel("Resolved branches (%)")
plt.xticks([0,2,4,6,8,10])
plt.yticks([60,70,80,90,100])
plt.ylim(55,105)
plt.xlabel("Embryonic time (days)")
save_plot(plots_path / "e9.5_marked_branches.svg", fig)

## PE heritability

In [ ]:
clone_tdata = tdata[tdata.obs.clone == "E9.5-R1-C1"].copy()
clone_tdata.obs["rank"] = np.arange(0, len(clone_tdata))
clone_tdata.obs["zoom"] = clone_tdata.obs.eval("60000 < rank < 60500")

example tree

In [ ]:
fig, ax = plt.subplots(figsize=(.8, 2.5))
py.pl.branches(clone_tdata, tree = "E9.5-R1-C1", depth_key = "time", ax = ax, linewidth = .4)
py.pl.annotation(clone_tdata,keys = "pe_expr",vmin = 0, vmax = 4, width = 0.25, ax = ax, label = False)
py.pl.annotation(clone_tdata,keys = "edit_frac",vmin = .3, vmax = .7, width = 0.25, ax = ax, label = False, cmap = "magma")
py.pl.annotation(clone_tdata, keys = ["zoom"], ax = ax, label = False)
save_plot(plots_path / "example_tree_pe_expr.svg", rasterize = True)

Zoom in

In [ ]:
fig, ax = plt.subplots(figsize=(.8, 2.5))
py.pl.branches(clone_tdata[clone_tdata.obs.zoom == True], tree = "E9.5-R1-C1", depth_key = "time", ax = ax, linewidth = .4) 
py.pl.annotation(clone_tdata,keys = "pe_expr",vmin = 0, vmax = 4, width = 0.2, ax = ax, label = False)
py.pl.annotation(clone_tdata,keys = "edit_frac",vmin = .3, vmax = .7, width = 0.25, ax = ax, label = False, cmap = "magma")
save_plot(plots_path / "example_tree_zoom_pe_expr.svg", rasterize = True)

Moran's I

In [ ]:
py.tl.tree_neighbors(clone_tdata, n_neighbors=5, depth_key="time", update = False)
autocorr = py.tl.autocorr(clone_tdata, keys="pe_expr", copy = True)
print("PE expression autocorrelation:",autocorr.autocorr.values[0])

## PE expr vs edit frac

In [ ]:
clone_tdata.obs["edit_pct"] = clone_tdata.obs["edit_frac"] * 100
fig, ax = plt.subplots(figsize=(2, 2))
sns.regplot(data=clone_tdata.obs.sample(10000),  x = "pe_expr", y = "edit_pct", scatter = False, line_kws={"color": "black"})
sns.scatterplot(data=clone_tdata.obs.sample(10000),  x = "pe_expr", y = "edit_pct", alpha = 0.1, s = 5, 
                hue = "edit_pct", palette = "magma", hue_norm = mcolors.Normalize(vmin=30, vmax=70), legend=False)
plt.xlabel("PE2max expression")
plt.ylabel("Edit fraction (%)")
save_plot(plots_path / "pe_expr_vs_edit_frac.svg", rasterize = True)

## Edit fraction by phase

In [ ]:
fig, ax = plt.subplots(figsize=(2, 2))
sns.boxplot(data=clone_tdata.obs, x = "phase", y = "edit_pct", showfliers=False, hue = "phase",
            palette=phase_palette, ax = ax, legend = False, saturation = 1)
plt.xlabel("Phase")
plt.ylabel("Mean edit fraction (%)")
save_plot(plots_path / "edit_frac_by_phase.svg")

## Embryo statistics

Character statistics

In [ ]:
unique_leaves = {}
for embryo, df in tdata.obs.query('clone.notnull()').groupby("embryo"):
    embryo_characters = characters.loc[df.index].copy()
    embryo_characters = embryo_characters.replace("-",pd.NA)
    n_unique = embryo_characters.drop_duplicates().shape[0]
    unique_leaves[embryo] = n_unique/embryo_characters.shape[0]
pass_rate = tdata.obs.query("type == 'donor'").groupby("embryo")["clone"].apply(lambda x: x.notnull().mean())
edit_rate = (
    tdata.obs.query("clone.notnull()")
    .groupby("embryo")["edit_frac"]
    .apply(lambda x: (x > 0.25).mean())
)

Topology statistics

In [ ]:
py.pp.add_depth(tdata)
def multifurcation_fraction(tree: nx.DiGraph) -> float:
    internal_nodes = [n for n in tree.nodes if tree.out_degree(n) > 0]
    binary_nodes = sum(tree.out_degree(n) == 2 for n in internal_nodes)
    return 1 - binary_nodes / len(internal_nodes)

topology_stats = []
for clone, tree in tdata.obst.items():
    edges = tree.number_of_edges()
    leaves = sum(1 for n in tree.nodes if tree.out_degree(n) == 0)
    topology_stats.append({
        "clone": clone,
        "marked_branches": edges,
        "unmarked_branches": (2 * leaves - 2) - edges,
        "frac_branches_resolved": edges / (2 * leaves - 2),
        "multifurcation_fraction": multifurcation_fraction(tree),
    })
topology_stats = pd.DataFrame.from_dict(topology_stats).set_index("clone")
tdata.obs["n"] = 1
topology_stats["clone_size"] = tdata.obs.groupby("clone")["n"].sum()
topology_stats["embryo"] = topology_stats.index.str.split("-C").str[0]

topology_stats = topology_stats.groupby("embryo").agg(
    marked_branches=("marked_branches", "sum"),
    unmarked_branches=("unmarked_branches", "sum"),
    frac_branches_resolved=(
        "frac_branches_resolved",
        lambda x: np.average(x,weights=topology_stats.loc[x.index, "clone_size"]),
    ),
    multifurcation_fraction=(
        "multifurcation_fraction",
        lambda x: np.average(x,weights=topology_stats.loc[x.index, "clone_size"]),
    ),
)

Total cells

In [ ]:
total_cells =  {
    "E7.5-R1": 3825,
    "E7.5-R2": 6750,
    "E7.5-R3": 7687,
    "E8.0-R1": 14375,
    "E8.0-R2": 19375,
    "E8.0-R3": 12500,
    "E8.5-R1": 55000,
    "E8.5-R2": 91250,
    "E8.5-R3": 46250,
    "E9.0-R1": 187500,
    "E9.0-R2": 256250,
    "E9.0-R3": 134375,
    "E9.5-R1": 759360,
    "E9.5-R2": 696864,
    "E9.5-R3": 510000,
    "E10.0-R1": 971875,
}

Format statistics

In [ ]:
embryo_stats = tdata.obs.groupby("embryo").agg(
    n_cells = ("n", "sum"),
    n_tracing = ("clone", lambda x: x.notna().sum()),
    n_captures = ("capture", "nunique"),
    mean_umis = ("total_counts", "mean"),
    n_clones = ("clone","nunique"),
    chimerism_rate = ("type", lambda x: (x.dropna() == "donor").mean()),
    mean_detection_rate = ("detection_rate", "mean"),
    mean_edit_frac = ("edit_frac", "mean"),
    mean_tree_depth = ("depth", "mean"),
)
embryo_stats = embryo_stats.loc[embryos,:].copy()
embryo_stats["n_marked_branches"] = topology_stats["marked_branches"]
embryo_stats["n_unmarked_branches"] = topology_stats["unmarked_branches"]
embryo_stats["frac_branches_resolved"] = topology_stats["frac_branches_resolved"]
embryo_stats["uniquely_marked_leaves"] = embryo_stats.index.map(unique_leaves)
embryo_stats["cell_total_estimate"] = embryo_stats.index.map(total_cells)
embryo_stats["cell_recovery_estimate"] = embryo_stats["n_cells"] / embryo_stats["cell_total_estimate"]
embryo_stats["frac_detection_>75%"] = pass_rate
embryo_stats["frac_edited_>25%"] = edit_rate
embryo_stats.round(2).to_csv(results_path / "lineage_stats.csv", index=True)

In [ ]:
py.pp.add_depth(tdata)